## 4.	Perform matrix factorization using SVD on a recommendation dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.stats import pearsonr

In [2]:
ratings = pd.read_csv("C:\\Users\\HP-LSC-086\Desktop\\Goura Nachika\\Experiment 4\\ratings.csv")   # <-- change the file name if needed

print("Raw Dataset:")
display(ratings)

Raw Dataset:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [3]:
# 2. Create User–Item Rating Matrix
ratings_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

print("\nUser–Item Rating Matrix:")
display(ratings_matrix.head())


User–Item Rating Matrix:


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# Convert to NumPy array for SVD
R = ratings_matrix.values

In [5]:
# 3. Perform SVD Decomposition
U, sigma, Vt = np.linalg.svd(R, full_matrices=False)

print("\nShapes:")
print("U:", U.shape)
print("Sigma:", sigma.shape)
print("Vt:", Vt.shape)


Shapes:
U: (610, 610)
Sigma: (610,)
Vt: (610, 9724)


In [6]:
# 4. Keep Top K Latent Factors (optional)
k = 50    # number of latent features to keep
sigma_k = np.diag(sigma[:k])
U_k = U[:, :k]
Vt_k = Vt[:k, :]

print("\nReduced matrices:")
print("U_k:", U_k.shape)
print("Sigma_k:", sigma_k.shape)
print("Vt_k:", Vt_k.shape)


Reduced matrices:
U_k: (610, 50)
Sigma_k: (50, 50)
Vt_k: (50, 9724)


In [7]:
# 5. Reconstruct Approximate Rating Matrix
R_pred = np.dot(np.dot(U_k, sigma_k), Vt_k)

predicted_ratings = pd.DataFrame(
    R_pred,
    index=ratings_matrix.index,
    columns=ratings_matrix.columns
)

print("\nPredicted Ratings Matrix (Approximation):")
display(predicted_ratings.head())


Predicted Ratings Matrix (Approximation):


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,2.181872,0.393674,0.838186,-0.082365,-0.546279,2.521662,-0.887231,-0.025221,0.196969,1.606758,...,-0.024984,-0.021415,-0.028553,-0.028553,-0.024984,-0.028553,-0.024984,-0.024984,-0.024984,-0.058988
2,0.209809,0.004821,0.030742,0.017252,0.183764,-0.060660,0.083306,0.023797,0.048100,-0.151968,...,0.018895,0.016196,0.021594,0.021594,0.018895,0.021594,0.018895,0.018895,0.018895,0.031966
3,0.013394,0.034726,0.050525,0.000200,-0.005577,0.114673,-0.007461,0.000738,0.004747,-0.061284,...,-0.001612,-0.001382,-0.001843,-0.001843,-0.001612,-0.001843,-0.001612,-0.001612,-0.001612,-0.000530
4,2.012072,-0.394882,-0.290386,0.093864,0.123312,0.259765,0.472667,0.035965,0.011293,-0.021983,...,0.001966,0.001685,0.002247,0.002247,0.001966,0.002247,0.001966,0.001966,0.001966,-0.021462
5,1.336714,0.772954,0.064577,0.113880,0.274994,0.584480,0.251048,0.131534,-0.086310,1.035361,...,-0.004407,-0.003778,-0.005037,-0.005037,-0.004407,-0.005037,-0.004407,-0.004407,-0.004407,-0.006099


In [8]:
# 6. Predict Top Movie Suggestions for a User
def recommend_movies(user_id, n=10):
    user_row = predicted_ratings.loc[user_id]
    
    # Movies already rated by the user
    rated_movies = ratings[ratings["userId"] == user_id]["movieId"].tolist()
    
    # Remove rated movies from recommendations
    user_row = user_row.drop(rated_movies)
    
    # Top n recommendations
    return user_row.sort_values(ascending=False).head(n)

# Example: recommendations for userId = 1
print("\nTop Recommendations for user 1:")
display(recommend_movies(1, 10))


Top Recommendations for user 1:


movieId
1036    4.009045
1221    3.303897
1387    3.303409
1968    2.869847
858     2.859630
1259    2.785549
2804    2.603560
2080    2.455787
4011    2.403890
2081    2.372148
Name: 1, dtype: float64